# Figure 1: Comparative Model Performance Visualization

## Purpose
Generate **comparative visualization** of forecasting, nowcasting, and contemporaneous model performance across all three modeling approaches.

## Methodology
1. **Load Results**: Import R² values from forecasting, nowcasting, and contemporaneous models
2. **Create Comparison Charts**: Generate side-by-side actual vs. predicted scatter plots with regression lines for each approach
3. **Calculate Metrics**: Compute and display R², accuracy, sensitivity, and precision across models
4. **Visualize Performance**: Create bar chart (accuracy comparison) and scatter plot (sensitivity vs. precision trade-off)

## Expected Outputs
- **nature_style_comparison_yellow_fit_textR2.jpg**: Three-panel scatter plot showing predicted vs. actual values with R² for each model
- **accuracy_barchart_nature.jpg**: Bar chart comparing accuracy across three approaches
- **precision_recall_scatter.jpg**: Scatter plot of sensitivity vs. precision trade-offs between models

## Note on Paths
**Execution note**: All input CSV files must be pre-generated by their respective main notebooks

Required input files (generated by other notebooks):
- `r2_frame_forecasting.csv` - Generated by Table1_Forecasting_main.ipynb
- `r2_frame_nowcasting.csv` - Generated by Table1_Nowcasting_two_layer.ipynb  
- `r2_frame_cv.csv` - Generated by Table1_Contemporaneous_main.ipynb

Released path construction:
```python
import os
data_dir = 'produced_graph'
r2_forecasting = pd.read_csv(os.path.join(data_dir, 'r2_frame_forecasting.csv'))
r2_nowcasting = pd.read_csv(os.path.join(data_dir, 'r2_frame_nowcasting.csv'))
r2_cs = pd.read_csv(os.path.join(data_dir, 'r2_frame_cv.csv'))
```

## Dependencies
- pandas, numpy, matplotlib, seaborn, scikit-learn

In [ ]:
import pandas as pd
import numpy as np  



# read csv file
r2_nowcasting = pd.read_csv(r'produced_graph/r2_frame_nowcasting.csv')
r2_forecasting = pd.read_csv(r'produced_graph/r2_frame_forecasting.csv')
r2_cs = pd.read_csv(r'produced_graph/r2_frame_cv.csv')





In [ ]:
# rename phase3_pred_nc as Predicted, phase3_test_nc as Actual
r2_nowcasting.rename(columns={'phase3_pred_nc': 'Predicted', 'phase3_test_nc': 'Actual'}, inplace=True)
r2_forecasting.rename(columns={'phase3_pred_fc': 'Predicted', 'phase3_test_fc': 'Actual'}, inplace=True)
r2_cs.rename(columns={'phase3_pred_cv': 'Predicted', 'phase3_test_cv': 'Actual'}, inplace=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
import numpy as np


In [ ]:

# --- Configuration ---

# **IMPORTANT**: Replace these with your actual column names if different
actual_col = 'Actual'
predicted_col = 'Predicted'


# List of dataframes and their titles
datasets = [
    (r2_forecasting, 'Forecasting'),
    (r2_nowcasting, 'Nowcasting'),
    (r2_cs, 'Contemporaneous')
]

# --- Styling (Approximate Nature Style) ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'axes.linewidth': 1.0,
    'grid.color': 'lightgray',
    'grid.linestyle': '--',
    'grid.linewidth': 0.5,
    'lines.markersize': 5, # Note: scatter marker size is set directly in ax.scatter
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': 'black',
    'savefig.dpi': 300
})

# Define a consistent color palette (Yellow/Orange Scheme)
color_scatter = '#ff7f0e' # Orange (closer to yellow scheme than blue)
# Alternative Yellow: '#FFC107' # Amber/Gold
color_regression = '#d62728' # Red for regression line (good contrast)
color_identity = 'grey'     # Grey for y=x line

# --- Plotting ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), sharex=True, sharey=True) # Increased height slightly for text

all_values = [] # To determine overall axis limits

for i, (df, title) in enumerate(datasets):
    ax = axes[i]

    # Check if columns exist
    if actual_col not in df.columns or predicted_col not in df.columns:
        print(f"Warning: Columns '{actual_col}' or '{predicted_col}' not found in DataFrame for '{title}'. Skipping.")
        ax.set_title(f"{title}\n(Data Missing)")
        ax.text(0.5, 0.5, "Data columns not found", horizontalalignment='center', verticalalignment='center', transform=ax.transAxes)
        continue

    # Extract data and handle potential NaNs
    df_clean = df[[actual_col, predicted_col]].dropna()
    if df_clean.empty:
        print(f"Warning: No valid data points after dropping NaNs for '{title}'. Skipping.")
        ax.set_title(f"{title}\n(No Valid Data)")
        ax.text(0.5, 0.5, "No valid data points", horizontalalignment='center', verticalalignment='center', transform=ax.transAxes)
        continue

    y_actual = df_clean[actual_col]
    X_predicted = df_clean[[predicted_col]]
    all_values.extend(y_actual.tolist())
    all_values.extend(X_predicted[predicted_col].tolist())

    # Calculate R^2 score
    r2 = r2_score(y_actual, X_predicted)

    # Scatter plot: Actual vs Predicted (Orange/Yellow scheme)
    # Using s=20, alpha=0.7 like the first plot attempt for scatter appearance
    ax.scatter(X_predicted, y_actual, alpha=0.7, label='Data Points', s=20, c=color_scatter, edgecolors='w', linewidth=0.5)

    # --- Add Linear Regression Line ---
    model = LinearRegression()
    model.fit(X_predicted, y_actual)
    x_line = np.array([X_predicted.min(), X_predicted.max()]).reshape(-1, 1)
    y_line = model.predict(x_line)

    # Plot the regression line (Label does NOT include R2 now)
    ax.plot(x_line, y_line, color=color_regression, lw=2, label='Best Fit')
    # --- End of Regression Line Addition ---

    # Plot y=x line (Perfect Prediction) - Added later for consistent limits

    # --- Add R^2 text annotation separately ---
    # Determine position based on axis limits (calculated later)
    # We'll store r2 and add text after setting limits

    # Set labels and title
    ax.set_xlabel('Predicted Values')
    ax.set_ylabel('Actual Values' if i == 0 else '')
    ax.set_title(title)
    ax.grid(True)

    # Store r2 score for adding text later
    ax.r2_score_value = r2


# Determine overall axis limits AFTER processing all data
if not all_values:
    print("Error: No data found in any DataFrame to determine axis limits.")
    global_min, global_max = 0, 1 # Default fallback
else:
    val_min = min(all_values)
    val_max = max(all_values)
    range_val = val_max - val_min if val_max > val_min else 1.0
    global_min = val_min - range_val * 0.05
    global_max = val_max + range_val * 0.05

# Plot y=x line, set limits, and add R2 text for all subplots
for ax in axes:
    # Only proceed if axis limits are valid
    if global_min < global_max:
        # Set limits first
        ax.set_xlim(global_min, global_max)
        ax.set_ylim(global_min, global_max)
        ax.set_aspect('equal', adjustable='box')

        # Plot y=x line within the final limits
        plot_lim_min = max(global_min, global_min)
        plot_lim_max = min(global_max, global_max)
        ax.plot([plot_lim_min, plot_lim_max], [plot_lim_min, plot_lim_max], ls='--', color=color_identity, lw=1.5, label='Perfect Prediction (y=x)', zorder=1)

        # Add R^2 text annotation inside the plot area (top-left)
        if hasattr(ax, 'r2_score_value'): # Check if R2 was calculated for this ax
             text_x = global_min + (global_max - global_min) * 0.05 # 5% from left edge
             text_y = global_max - (global_max - global_min) * 0.05 # 5% from top edge
             ax.text(text_x, text_y, f'$R^2 = {ax.r2_score_value:.3f}$', fontsize=11, verticalalignment='top', horizontalalignment='left')

    else: # Handle case where limits were not determined
         ax.set_xlim(0, 1)
         ax.set_ylim(0, 1)


# Add a legend
handles, labels = [], []
# Get handles/labels from the first axis (they should be the same for all)
# Ensure axis has content before getting legend elements
if axes[0].has_data():
    h, l = axes[0].get_legend_handles_labels()
    handles.extend(h)
    labels.extend(l)

# Create a unique legend using dictionary keys
by_label = dict(zip(labels, handles))
# Reorder legend handles/labels if needed (optional)
ordered_labels = ['Data Points', 'Best Fit', 'Perfect Prediction (y=x)']
ordered_handles = [by_label[label] for label in ordered_labels if label in by_label]

fig.legend(ordered_handles, ordered_labels, loc='upper center', bbox_to_anchor=(0.5, 0.03), ncol=3, frameon=False)

fig.text(0.5, -0.1, r'(C) Predictive $R^2$ of Population Percentage falling in Phase 3 and Above',
         ha='center', fontsize=11, fontweight='bold')# Improve layout
plt.tight_layout(rect=[0, 0.05, 1, 0.98]) # Adjust rect to make space for legend/titles

# add figure title

# Show the plot
plt.show()

# To save the figure:
fig.savefig('produced_graph/nature_style_comparison_yellow_fit_textR2.jpg', dpi=250, bbox_inches='tight')
# fig.savefig('produced_graph/nature_style_comparison_yellow_fit_textR2.pdf', bbox_inches='tight')
# fig.savefig('produced_graph/nature_style_comparison_yellow_fit_textR2.svg', bbox_inches='tight')

In [ ]:

# --- Data ---
categories = ['Forecasting','Nowcasting', 'Contemporaneous']
accuracy_scores = [ 0.6495726495726496, 0.6538461538461539,0.698079]

# --- Styling (Refined Nature Style Approximation) ---
# Start with a base style that's clean
plt.style.use('seaborn-v0_8-whitegrid') # Provides a grid baseline we can customize
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'], # Standard fonts
    'font.size': 10,                       # Base font size
    'axes.labelsize': 10,                  # Axis label size
    'axes.titlesize': 12,                  # Title size
    'xtick.labelsize': 10,                 # X-tick label size
    'ytick.labelsize': 9,                  # Y-tick label size
    'axes.linewidth': 1.0,                 # Axis line width
    'figure.facecolor': 'white',           # White figure background
    'axes.facecolor': 'white',             # White axes background
    'axes.edgecolor': 'black',             # Black axis lines
    'grid.color': '#cccccc',               # Lighter grey for gridlines
    'grid.linestyle': '--',                # Dashed gridlines
    'grid.linewidth': 0.5,                 # Thinner gridlines
    'savefig.dpi': 300,                    # High resolution saving
    'axes.spines.top': False,              # Remove top axis line
    'axes.spines.right': False,            # Remove right axis line
    'xtick.major.size': 3,                 # Size of X axis ticks
    'ytick.major.size': 3,                 # Size of Y axis ticks
    'xtick.major.width': 1.0,              # Width of X axis ticks
    'ytick.major.width': 1.0,              # Width of Y axis ticks
    'xtick.direction': 'out',              # Ticks point outwards
    'ytick.direction': 'out',              # Ticks point outwards
})

# --- Plotting ---
fig, ax = plt.subplots(figsize=(5, 4.5)) # Adjusted size slightly

# Use a single, professional color for bars (e.g., grey or blue)
bar_color = '#808080' # Medium Grey

# Create the bar chart
bars = ax.bar(categories, accuracy_scores, color=bar_color, width=0.7) # Slightly adjust width if needed

# Add labels and title
ax.set_ylabel('Accuracy Score', fontsize=10) # Explicitly set label size

fig.text(0.5, -0.05, '(A) Predictive Accuracy of overall IPC Phases',
         ha='center', fontsize=10, fontweight='bold')# Improve layout


# Set y-axis limits to better show differences, ensure some space
min_val = min(accuracy_scores)
max_val = max(accuracy_scores)
# Start just below the lowest bar value if not starting at 0
ax.set_ylim(bottom=min_val * 0.90, top=max_val * 1.05)
# ax.set_ylim(bottom=0) # Uncomment to force y-axis start at 0

# Add the accuracy values on top of the bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + (max_val * 0.005), # Add small offset above bar
             f'{yval:.3f}', # Format to 3 decimal places
             va='bottom', ha='center', fontsize=9) # Position text slightly above bar

# Customize grid lines (make y-grid lighter, remove x-grid)
ax.yaxis.grid(True, linestyle='--', which='major', color='#cccccc', alpha=0.7) # Lighter grid
ax.xaxis.grid(False) # No vertical grid lines

# Ensure axis ticks point outwards (already set in rcParams, but can be explicit)
# ax.tick_params(axis='both', direction='out')

# Improve layout
plt.tight_layout()

# Show the plot
plt.show()

# To save the figure:
# fig.savefig('produced_graph/accuracy_barchart_nature.png', dpi=300, bbox_inches='tight')
fig.savefig('produced_graph/accuracy_barchart_nature.jpg', bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Data ---
models = ['Forecasting','Nowcasting','Contemporaneous']
sensitivity_recall = np.array([0.9408418657565415,0.9419795221843004,0.908146] )
precision = np.array([0.7750702905342081,0.7774647887323943,0.80132])

# --- Styling (Nature Style Approximation) ---
plt.style.use('seaborn-v0_8-whitegrid') # Start with a clean base style
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'], # Standard fonts
    'font.size': 10,                       # Base font size
    'axes.labelsize': 10,                  # Axis label size
    'axes.titlesize': 10,                  # Title size
    'xtick.labelsize': 10,                  # X-tick label size
    'ytick.labelsize': 10,                  # Y-tick label size
    'legend.fontsize': 10,                  # Legend font size
    'axes.linewidth': 1.0,                 # Axis line width
    'figure.facecolor': 'white',           # White figure background
    'axes.facecolor': 'white',             # White axes background
    'axes.edgecolor': 'black',             # Black axis lines
    'grid.color': '#cccccc',               # Lighter grey for gridlines
    'grid.linestyle': '--',                # Dashed gridlines
    'grid.linewidth': 0.5,                 # Thinner gridlines
    'savefig.dpi': 300,                    # High resolution saving
    'axes.spines.top': False,              # Remove top axis line
    'axes.spines.right': False,            # Remove right axis line
    'xtick.major.size': 3,                 # Size of X axis ticks
    'ytick.major.size': 3,                 # Size of Y axis ticks
    'xtick.major.width': 1.0,              # Width of X axis ticks
    'ytick.major.width': 1.0,              # Width of Y axis ticks
    'xtick.direction': 'out',              # Ticks point outwards
    'ytick.direction': 'out',              # Ticks point outwards
    'scatter.marker': 'o',                 # Default marker shape
    'lines.markersize': 7,                 # Default marker size (use 's' in scatter)
})

# --- Plotting ---
fig, ax = plt.subplots(figsize=(6, 5)) # Adjust figure size as needed

# Define colors and markers for differentiation
colors = ['#1f77b4', '#ff7f0e', '#2ca02c'] # Blue, Orange, Green
markers = ['o', 's', '^'] # Circle, Square, Triangle

# Plot each point
for i, model in enumerate(models):
    ax.scatter(sensitivity_recall[i], precision[i],
               color=colors[i],
               marker=markers[i],
               s=80,  # Marker size
               label=model,
               alpha=0.8,
               edgecolors='w', # Slight white edge for clarity
               linewidth=0.5)

# Add labels and title
ax.set_xlabel('Sensitivity (Recall)')
ax.set_ylabel('Precision')
fig.text(0.5, -0.05, r'(B) Sensitivity vs Precision of Phase 3 and Above',
         ha='center', fontsize=11, fontweight='bold')# Improve layout

# Adjust axis limits - provide some padding around the data points
x_min, x_max = sensitivity_recall.min(), sensitivity_recall.max()
y_min, y_max = precision.min(), precision.max()
x_pad = (x_max - x_min) * 0.15 # 15% padding
y_pad = (y_max - y_min) * 0.15

# Ensure padding doesn't push limits below reasonable values (e.g., 0)
ax.set_xlim(max(0, x_min - x_pad), min(1, x_max + x_pad)) # Limits between 0 and 1
ax.set_ylim(max(0, y_min - y_pad), min(1, y_max + y_pad)) # Limits between 0 and 1

# Add legend
ax.legend(title='Model', frameon=False, loc='lower left') # Place legend, e.g., lower left

# Customize grid lines (optional: make them lighter or remove)
ax.grid(True, linestyle='--', which='major', color='#cccccc', alpha=0.7)

# Improve layout
plt.tight_layout()

# Show the plot
plt.show()

# To save the figure:
# fig.savefig('produced_graph/precision_recall_scatter.png', dpi=300, bbox_inches='tight')
fig.savefig('produced_graph/precision_recall_scatter.jpg', dpi=230, bbox_inches='tight')